<a id="top"></a>

# Demo: one wired round-trip, end to end

```yaml
title: "Demo: one wired round-trip, end to end"
keywords: round-trip, dispatch, ToolMessage, tool_call_id, wire a tool, name description schema, teacher demo
estimated duration: 12
```

> **Lesson:** L07. Demo 2 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L07/demos_or_activities.md). Makes **live** Claude calls — set `ANTHROPIC_API_KEY` before running. Clear outputs before committing.
>
> **The client is LangChain's `ChatAnthropic`** (introduced in L03). `bind_tools` makes the plain `calculator` function a tool; the model's `AIMessage.tool_calls` is the request, and you hand the result back as a **`ToolMessage`** that names the same call id. The API key still loads through the config seam (`require_anthropic_key`); never hard-coded.
>
> The point to land: wiring a tool is **name -> describe -> schema -> dispatch -> result -> continue**. The application validated and ran the function; the model only proposed.

## Contents

- [1. Setup](#1-setup)
- [2. First turn — the model proposes a call](#2-first-turn--the-model-proposes-a-call)
- [3. Dispatch — the application runs the function](#3-dispatch--the-application-runs-the-function)
- [4. Continue — hand the result back as a ToolMessage](#4-continue--hand-the-result-back-as-a-toolmessage)
- [5. The tool closed the accuracy gap](#5-the-tool-closed-the-accuracy-gap)
- [6. Takeaways](#6-takeaways)

## 1. Setup

Reuse Demo 1's client, tool, and prompt so the technique completes on something familiar.

In [ ]:
from typing import Annotated

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import BaseMessage, HumanMessage, ToolMessage

from fluffy_potato_curriculum.common.config import require_anthropic_key

SONNET = "claude-sonnet-4-6"


# The ONE tool that carries all four demos: a deterministic calculator.
# L07 is deliberately single-tool, single-round-trip (multi-tool is L08, the
# agent loop is L10). Resist adding a second tool. bind_tools infers the tool
# definition (name, docstring, arg schema) from this function -- see Demo 1.
def calculator(
    expression: Annotated[str, "The arithmetic expression to evaluate, e.g. '18374 * 92431'."],
) -> str:
    """Evaluate a basic arithmetic expression (the four operators and parentheses) and
    return the exact numeric result. Use this whenever the user asks for the result of
    a calculation.

    Restricted to digits and the operators + - * / ( ) . and whitespace so a
    hallucinated expression cannot run arbitrary code. Raises ValueError on
    anything else -- that rejection is the whole point of Demo 4."""
    allowed = set("0123456789+-*/(). ")
    if not expression or set(expression) - allowed:
        raise ValueError(f"refusing to evaluate non-arithmetic expression: {expression!r}")
    # eval is safe here ONLY because the character whitelist above blocks names,
    # attributes, and calls. Never eval untrusted input without such a guard.
    return str(eval(expression))


model = ChatAnthropic(model=SONNET, api_key=require_anthropic_key(), max_tokens=400)
model_with_tools = model.bind_tools([calculator])

PROMPT = "What is 18,374 multiplied by 92,431?"
print("setup ready (live model:", SONNET, ")")

[↑ Back to top](#top)

## 2. First turn — the model proposes a call

The same first turn as Demo 1: the model returns an `AIMessage` carrying a tool call. Last time you stopped here.

In [ ]:
messages: list[BaseMessage] = [HumanMessage(PROMPT)]
first = model_with_tools.invoke(messages)
call = first.tool_calls[0]
print("model proposed:", call["name"], call["args"], "id=", call["id"])

[↑ Back to top](#top)

## 3. Dispatch — the application runs the function

Now the step Demo 1 skipped: match the tool name, pull the `expression` argument, run the **real** function, and capture the result. *This* number is computed, not generated.

In [ ]:
assert call["name"] == "calculator"  # dispatch on the name the model asked for
args = call["args"]  # already a parsed dict, not a string blob
result = calculator(args["expression"])  # the application runs the tool
print("application matched tool  :", call["name"])
print("application extracted args:", args)
print("calculator(...) returned  :", result, "  <- a REAL computed number")

[↑ Back to top](#top)

## 4. Continue — hand the result back as a ToolMessage

Build the next message: a **`ToolMessage`** (a dedicated tool-role message) whose `tool_call_id` references the same call **id**, carrying the function's output. Record the assistant's tool-requesting turn first, then send the *full* conversation back. The model returns a final answer that uses the real number.

In [ ]:
# 1) record the assistant's tool-requesting turn, 2) add the result as a ToolMessage.
messages.append(first)
messages.append(ToolMessage(content=result, tool_call_id=call["id"]))

# model_with_tools still carries the tool definition, so it rides along AGAIN on
# this second call -- the model is stateless; the schema is in every prompt.
final = model_with_tools.invoke(messages)
print("final answer:\n", final.content)

[↑ Back to top](#top)

## 5. The tool closed the accuracy gap

Put Demo 1's tool-free guess next to the calculator's real answer — same model, same question.

In [ ]:
true_value = 18374 * 92431
print("ground truth (Python)      :", true_value)
print("calculator tool result     :", result)
print("model's final answer used it and is now correct.")

[↑ Back to top](#top)

## 6. Takeaways

- The five mechanical steps, in one pass: **name** the function, **describe** it, give it a **schema**, **dispatch** on the returned call, run the **result**, **continue** the conversation. That is the whole Objective-1 skill.
- The **application** validated the arguments and ran the function. The model proposed; the application disposed.
- The result went back as a **`ToolMessage`** — a dedicated tool-role message whose `tool_call_id` names the request it answers. Your code, not the human, produced it. This sets up [Demo 3](L0706_lecture.ipynb)'s message-by-message trace.
- The tool definition rode along on the *second* call too: the model is **stateless**, so the schema is in every prompt.

[↑ Back to top](#top)